# Example 1: Load dataset, rename idents, run enrichment and module scoring, save dataset

This notebook shows the minimal API workflow using `scomnom` only.


In [ ]:
from pathlib import Path
import scomnom as om

# Edit these paths for your environment
in_path = Path("/path/to/input_dataset.zarr")
out_path = Path("/path/to/output_dataset_renamed.zarr")


In [ ]:
adata = om.load_dataset(in_path)
adata


## Build a `Cnn -> new_label` mapping
Use short `Cnn` keys only (for example `C00`, `C01`).

In [ ]:
rename_mapping = {
    "C00": "Hepatocyte",
    "C01": "Endothelial",
    "C02": "Immune",
}

new_round_id = om.adata_ops.rename_idents(
    adata,
    mapping=rename_mapping,
    round_name="manual_rename_api",
    collapse_same_labels=False,
    update_existing_round=False,
    set_active=True,
)
print(f"Created rename round: {new_round_id}")


## Run round-native enrichment in memory

This updates the selected round on `adata` in place and returns the stored decoupler payload.

In [ ]:
decoupler_payload = om.markers_and_de.enrichment_cluster(
    adata,
    round_id=new_round_id,
    condition_key=None,
    gene_filter=("not gene.str.startswith('MT-')",),
    run_progeny=True,
    run_dorothea=True,
)
list(decoupler_payload.keys())


In [ ]:
om.plotting.plot_decoupler_payload(
    decoupler_payload,
    net_name="msigdb",
    display=True,
)


## Run enrichment from exported DE tables

This mode does not need an AnnData object. Point it at an exported DE tables directory and it returns a nested payload bundle.

In [ ]:
de_tables_dir = Path("/path/to/results/tables/de_r5_archetypes_round1")
de_enrichment = om.markers_and_de.enrichment_de_from_tables(
    de_tables_dir,
    gene_filter=("not gene.str.startswith('MT-')",),
    de_decoupler_source="auto",
)
list(de_enrichment.keys())


In [ ]:
first_condition = next(iter(de_enrichment))
first_contrast = next(iter(de_enrichment[first_condition]))
first_source_payload = next(iter(de_enrichment[first_condition][first_contrast].values()))

om.plotting.plot_de_decoupler_payload(
    first_source_payload,
    net_name="msigdb",
    display=True,
)


## Run module scoring in memory

This scores user-defined modules per cell, stores the summary on the selected round, and returns the stored module-score payload.


In [ ]:
module_score_payload = om.markers_and_de.module_score(
    adata,
    module_files=(Path("/path/to/signatures.gmt"),),
    module_set_name="curated_programs",
    round_id=new_round_id,
    condition_key="sex:MASLD",
    module_score_method="scanpy",
)
module_score_payload["summary_mean_z"].head()


In [ ]:
om.plotting.plot_module_score_summary_heatmap(
    module_score_payload["summary_mean_z"],
    title="Curated module scores",
    display=True,
)


In [ ]:
om.save_dataset(adata, out_path, fmt="zarr", archive=True)
print(f"Saved: {out_path}")
